In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 15.0 MB/s eta 0:00:00


This code loads the Pima Indians Diabetes dataset from a GitHub URL into a Pandas DataFrame and assigns appropriate column names. It prepares the data for further processing or modeling.

In [ ]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']
df=pd.read_csv(url, names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


This code segment replaces zeros in selected medical columns with NaN and fills the missing values with the column-wise mean to clean the dataset.

In [ ]:
col_with_missing_vols=['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[col_with_missing_vols]=df[col_with_missing_vols].replace(0, np.nan)
df.fillna(df.mean(), inplace=True)
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


This code segment splits the dataset into training and testing sets, then standardizes the features using StandardScaler to normalize the data before model training.

In [29]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']
x_train, x_test, y_train, y_test=train_test_split(x,y,test_size=0.3, random_state=42)
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.fit_transform(x_test)
print(f'Training dataset shape: {x_train.shape}')
print(f'Testing dataset shape:{ x_test.shape}')

Training dataset shape: (537, 8)
Testing dataset shape:(231, 8)


This code segment defines an Optuna objective function to optimize a RandomForestClassifier by tuning n_estimators and max_depth using 5-fold cross-validation, with accuracy as the scoring metric.

In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
def objective(trial):
  n_estimators=trial.suggest_int('n_estimators', 50, 200)
  max_depth=trial.suggest_int('max_depth', 3, 20)
  model=RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )
  score=cross_val_score(model, x_train, y_train, cv=5, scoring='accuracy').mean()
  return score

This code segment creates and runs an Optuna study named 'akmal' to maximize the accuracy of a RandomForestClassifier. It uses the TPE sampler for smarter hyperparameter search over 50 trials.

In [36]:
from os import name
study= optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(), study_name='akmal')
study.optimize(objective, n_trials=50)


[I 2025-04-13 04:42:11,734] A new study created in memory with name: akmal
[I 2025-04-13 04:42:12,799] Trial 0 finished with value: 0.7541363793700242 and parameters: {'n_estimators': 126, 'max_depth': 4}. Best is trial 0 with value: 0.7541363793700242.
[I 2025-04-13 04:42:14,029] Trial 1 finished with value: 0.7559709241952233 and parameters: {'n_estimators': 134, 'max_depth': 8}. Best is trial 1 with value: 0.7559709241952233.
[I 2025-04-13 04:42:15,665] Trial 2 finished with value: 0.7503980616130148 and parameters: {'n_estimators': 195, 'max_depth': 4}. Best is trial 1 with value: 0.7559709241952233.
[I 2025-04-13 04:42:16,385] Trial 3 finished with value: 0.755988231221876 and parameters: {'n_estimators': 50, 'max_depth': 8}. Best is trial 3 with value: 0.755988231221876.
[I 2025-04-13 04:42:18,817] Trial 4 finished with value: 0.7708722741433022 and parameters: {'n_estimators': 172, 'max_depth': 16}. Best is trial 4 with value: 0.7708722741433022.
[I 2025-04-13 04:42:20,012] Tria

This code segment prints the best accuracy score achieved during the Optuna optimization and the corresponding hyperparameters of the best trial.

In [37]:
print(f'best trial accuracy: {study.best_trial.value}')
print(f'best trial paramteres: {study.best_trial.params}')

best trial accuracy: 0.7708722741433022
best trial paramteres: {'n_estimators': 172, 'max_depth': 16}


This code segment generates and displays a plot of the optimization history of the Optuna study, illustrating the progress of the objective function (e.g., accuracy) across all trials.

In [49]:
plot_optimization_history(study).show()

The code segment generates a parallel coordinate plot to visualize how the hyperparameters in the Optuna study affect the objective function's performance.

In [52]:
plot_parallel_coordinate(study).show()

The code plot_slice(study).show() generates a slice plot to visualize how changing one hyperparameter affects the objective function's performance, while keeping other hyperparameters fixed.

In [55]:
plot_slice(study).show()

The code segement generates a contour plot to visualize the relationship between two hyperparameters and the objective function's performance, helping to identify regions of optimal values for both hyperparameters.

In [56]:
plot_contour(study).show()

The code segment generates a parametric importance plot that visualizes the impact of each hyperparameter on the objective function, helping to identify which hyperparameters are most influential in the optimization process.

In [58]:
plot_param_importances(study).show()

This code trains a RandomForestClassifier using the best hyperparameters from the Optuna study, evaluates its performance on the test set, and prints the test accuracy. The accuracy is calculated by comparing predicted labels with actual labels.

In [41]:
from sklearn.metrics import accuracy_score
best_model=RandomForestClassifier(**study.best_trial.params, random_state=42)
best_model.fit(x_train, y_train)
y_pred=best_model.predict(x_test)
test_accuracy=accuracy_score(y_test, y_pred)
print(f'test accuracy with best hyperparameters:{test_accuracy:.2f}')

test accuracy with best hyperparameters:0.74


This code defines an Optuna objective function for optimizing a RandomForestClassifier. It tunes the n_estimators and max_depth hyperparameters and evaluates the model's performance using 5-fold cross-validation, returning the mean accuracy score.

In [42]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
def objective(trial):
  n_estimators=trial.suggest_int('n_estimators', 50, 200)
  max_depth=trial.suggest_int('max_depth', 3, 20)
  model=RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )
  score=cross_val_score(model, x_train,y_train, cv=5, scoring='accuracy').mean()
  return score

This code creates an Optuna study to maximize the accuracy of a RandomForestClassifier by optimizing hyperparameters using the RandomSampler. It runs the optimization process for 50 trials to find the best-performing hyperparameter configuration.

In [43]:
study=optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())
study.optimize(objective, n_trials=50)

[I 2025-04-13 05:06:06,760] A new study created in memory with name: no-name-44c922c0-ec72-4dfe-87d5-5528574106c4
[I 2025-04-13 05:06:10,084] Trial 0 finished with value: 0.7671339563862929 and parameters: {'n_estimators': 165, 'max_depth': 7}. Best is trial 0 with value: 0.7671339563862929.
[I 2025-04-13 05:06:12,018] Trial 1 finished with value: 0.7615610938040843 and parameters: {'n_estimators': 72, 'max_depth': 14}. Best is trial 0 with value: 0.7671339563862929.
[I 2025-04-13 05:06:14,295] Trial 2 finished with value: 0.7689685012114919 and parameters: {'n_estimators': 166, 'max_depth': 11}. Best is trial 2 with value: 0.7689685012114919.
[I 2025-04-13 05:06:14,862] Trial 3 finished with value: 0.757840083073728 and parameters: {'n_estimators': 62, 'max_depth': 5}. Best is trial 2 with value: 0.7689685012114919.
[I 2025-04-13 05:06:16,193] Trial 4 finished with value: 0.7634302526825891 and parameters: {'n_estimators': 149, 'max_depth': 7}. Best is trial 2 with value: 0.7689685012

This code prints the best training accuracy and the corresponding hyperparameters from the Optuna study's best trial. It helps identify the optimal model configuration based on the highest accuracy achieved.

In [44]:
print(f'best traiing accuracy: {study.best_trial.value}')
print(f'best trail parameters: {study.best_trial.params}')

best traiing accuracy: 0.770889581169955
best trail parameters: {'n_estimators': 177, 'max_depth': 9}


This code trains a RandomForestClassifier using the best hyperparameters from the Optuna study, makes predictions on the test set, calculates the accuracy, and prints the test accuracy.

In [45]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.76


This code imports various Optuna visualization functions to analyze the optimization study results. These plots help in understanding the relationship between hyperparameters and the objective function, as well as identifying the most important parameters.

In [46]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_param_importances, plot_contour

In [48]:
plot_optimization_history(study).show()

In [50]:
plot_parallel_coordinate(study).show()

In [53]:
plot_contour(study).show()

In [54]:
plot_slice(study).show()

In [57]:
plot_param_importances(study).show()

In [62]:
from optuna.visualization import plot_intermediate_values

This code imports three classifiers: RandomForestClassifier for ensemble learning, GradientBoostingClassifier for sequential model building, and SVC for support vector machine-based classification. These models are commonly used for various classification tasks.

In [63]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

This code utilizes Optuna for hyperparameter optimization of machine learning models, including SVM, RandomForest, and GradientBoosting. The objective function dynamically tunes the hyperparameters of the selected classifier, and the best model is evaluated on the test set to achieve the highest accuracy.

In [70]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

This code creates an Optuna study to maximize the model's performance by optimizing hyperparameters. The optimization process is performed for 100 trials to find the best combination of hyperparameters for the selected model.

In [71]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-04-13 05:51:22,692] A new study created in memory with name: no-name-de032ac7-9be0-4adc-9311-dcc9e7813ac1
[I 2025-04-13 05:51:22,743] Trial 0 finished with value: 0.7094972067039106 and parameters: {'classifier': 'SVM', 'C': 11.70467721598936, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 0 with value: 0.7094972067039106.
[I 2025-04-13 05:51:22,822] Trial 1 finished with value: 0.696461824953445 and parameters: {'classifier': 'SVM', 'C': 42.75292025824411, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 0 with value: 0.7094972067039106.
[I 2025-04-13 05:51:23,988] Trial 2 finished with value: 0.7690875232774674 and parameters: {'classifier': 'RandomForest', 'n_estimators': 258, 'max_depth': 14, 'min_samples_split': 8, 'min_samples_leaf': 5, 'bootstrap': False}. Best is trial 2 with value: 0.7690875232774674.
[I 2025-04-13 05:51:26,397] Trial 3 finished with value: 0.7616387337057727 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 290, 'learning_rate':

This code prints the accuracy of the best trial found during the optimization process and displays the hyperparameters associated with that trial. The best trial represents the optimal combination of hyperparameters for the selected model.

In [72]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best trial parameters: {study.best_trial.params}')

Best trial accuracy: 0.7895716945996275
Best trial parameters: {'classifier': 'SVM', 'C': 0.15068887109277518, 'kernel': 'linear', 'gamma': 'auto'}


The study.trials_dataframe() method in Optuna converts the trial results of the study into a pandas DataFrame. This DataFrame contains details about each trial, including the trial number, hyperparameters, objective values (e.g., accuracy), and other relevant metrics, making it easier to analyze and visualize the optimization process.

In [74]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.709497,2025-04-13 05:51:22.694589,2025-04-13 05:51:22.742772,0 days 00:00:00.048183,11.704677,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.696462,2025-04-13 05:51:22.743961,2025-04-13 05:51:22.822183,0 days 00:00:00.078222,42.752920,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.769088,2025-04-13 05:51:22.823762,2025-04-13 05:51:23.988489,0 days 00:00:01.164727,NaN,False,RandomForest,NaN,NaN,NaN,14.0,5.0,8.0,258.0,COMPLETE
3,3,0.761639,2025-04-13 05:51:23.989730,2025-04-13 05:51:26.396791,0 days 00:00:02.407061,NaN,NaN,GradientBoosting,NaN,NaN,0.012772,5.0,4.0,9.0,290.0,COMPLETE
4,4,0.739292,2025-04-13 05:51:26.398075,2025-04-13 05:51:31.015645,0 days 00:00:04.617570,NaN,NaN,GradientBoosting,NaN,NaN,0.020904,16.0,5.0,3.0,233.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.785847,2025-04-13 05:52:09.595913,2025-04-13 05:52:09.948584,0 days 00:00:00.352671,84.795631,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.785847,2025-04-13 05:52:09.950543,2025-04-13 05:52:10.005621,0 days 00:00:00.055078,0.212218,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.776536,2025-04-13 05:52:10.007393,2025-04-13 05:52:10.082899,0 days 00:00:00.075506,0.134738,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.789572,2025-04-13 05:52:10.084854,2025-04-13 05:52:10.138471,0 days 00:00:00.053617,0.132658,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


The code study.trials_dataframe()['params_classifier'].value_counts() counts the occurrences of each classifier type used during the trials in the Optuna study. It shows how many times each classifier (e.g., 'SVM', 'RandomForest', 'GradientBoosting') was selected during the hyperparameter optimization process.

In [75]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,71
RandomForest,20
GradientBoosting,9


The code study.trials_dataframe().groupby('params_classifier')['value'].mean() groups the trials by the classifier type (e.g., 'SVM', 'RandomForest', 'GradientBoosting') and calculates the mean objective value (such as accuracy) for each classifier. This allows you to compare the average performance of each model across all trials in the study.

In [76]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.741155
RandomForest,0.771974
SVM,0.777874
